# Notebook 9: Automated Model Comparison (PyCaret-Style)
## Car Price Prediction Project

**Objective:** Automatically compare 10+ regression models and find the best-performing one, then compare it against our manual models.

> **Note:** PyCaret does not support Python 3.12 (Colab's current version), so we replicate the same comparison using scikit-learn and XGBoost directly. This produces the same results — a ranked model comparison table.

---

## 9.1 Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install xgboost lightgbm catboost --quiet 2>/dev/null

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# All regression models to compare
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    BayesianRidge, HuberRegressor
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    AdaBoostRegressor, ExtraTreesRegressor,
    BaggingRegressor
)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

print("All libraries loaded successfully.")

## 9.2 Load and Prepare Data

In [ ]:
# Load the encoded dataset
df = pd.read_csv('/content/drive/MyDrive/car_price_encoded.csv')
print(f"Dataset shape: {df.shape}")

# Features and target
X = df.drop(columns=['price'])
y = df['price']

# Train-test split (same as preprocessing notebook)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaled version for linear models
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")

## 9.3 Define All Models

We compare **15+ models** — the same kind of comparison PyCaret would do.

In [ ]:
# Models that need scaling
scaled_models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=1.0, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=1.0, max_iter=5000),
    'Bayesian Ridge': BayesianRidge(),
    'Huber Regressor': HuberRegressor(max_iter=500),
    'SVR': SVR(kernel='rbf', C=100, epsilon=0.1),
    'KNN': KNeighborsRegressor(n_neighbors=5),
}

# Tree-based models (don't need scaling)
unscaled_models = {
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42),
    'Extra Trees': ExtraTreesRegressor(n_estimators=200, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42),
    'AdaBoost': AdaBoostRegressor(n_estimators=200, random_state=42),
    'Bagging': BaggingRegressor(n_estimators=200, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42, verbose=-1),
    'CatBoost': CatBoostRegressor(iterations=200, depth=5, learning_rate=0.1, random_seed=42, verbose=0),
}

print(f"Total models to compare: {len(scaled_models) + len(unscaled_models)}")

## 9.4 Train & Evaluate All Models

In [ ]:
results = []

print("Training all models...")
print("=" * 70)

# Train scaled models
for name, model in scaled_models.items():
    model.fit(X_train_scaled, y_train)
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)
    
    results.append({
        'Model': name,
        'Train R²': r2_score(y_train, y_train_pred),
        'Test R²': r2_score(y_test, y_test_pred),
        'Train RMSE': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'Train MAE': mean_absolute_error(y_train, y_train_pred),
        'Test MAE': mean_absolute_error(y_test, y_test_pred)
    })
    print(f"  ✓ {name:<25s} Test RMSE: ${results[-1]['Test RMSE']:>10,.0f}")

# Train unscaled models
for name, model in unscaled_models.items():
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    results.append({
        'Model': name,
        'Train R²': r2_score(y_train, y_train_pred),
        'Test R²': r2_score(y_test, y_test_pred),
        'Train RMSE': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'Train MAE': mean_absolute_error(y_train, y_train_pred),
        'Test MAE': mean_absolute_error(y_test, y_test_pred)
    })
    print(f"  ✓ {name:<25s} Test RMSE: ${results[-1]['Test RMSE']:>10,.0f}")

print("=" * 70)
print(f"All {len(results)} models trained!")

## 9.5 Model Comparison Table (Sorted by RMSE)

In [ ]:
# Create comparison table sorted by Test RMSE (best first)
comparison_df = pd.DataFrame(results).sort_values('Test RMSE').reset_index(drop=True)
comparison_df.index = comparison_df.index + 1  # Start ranking from 1
comparison_df.index.name = 'Rank'

print("\n" + "=" * 80)
print("  MODEL COMPARISON TABLE — Sorted by Test RMSE (Best First)")
print("=" * 80)
comparison_df

In [ ]:
# Highlight the best model
best_model_name = comparison_df.iloc[0]['Model']
best_test_rmse = comparison_df.iloc[0]['Test RMSE']
best_test_r2 = comparison_df.iloc[0]['Test R²']
best_test_mae = comparison_df.iloc[0]['Test MAE']

print(f"\n{'='*60}")
print(f"  BEST MODEL: {best_model_name}")
print(f"{'='*60}")
print(f"  Test R²:   {best_test_r2:.4f} ({best_test_r2*100:.1f}% variance explained)")
print(f"  Test RMSE: ${best_test_rmse:,.0f}")
print(f"  Test MAE:  ${best_test_mae:,.0f}")
print(f"  RMSE/Mean: {(best_test_rmse / y_test.mean()) * 100:.1f}% of average price")

## 9.6 Visual Comparison — Top 10 Models

In [ ]:
top10 = comparison_df.head(10)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(top10)))

# Test R²
axes[0].barh(range(len(top10)), top10['Test R²'], color=colors)
axes[0].set_yticks(range(len(top10)))
axes[0].set_yticklabels(top10['Model'])
axes[0].set_xlabel('Test R²')
axes[0].set_title('Test R² (higher = better)', fontweight='bold')
axes[0].invert_yaxis()
for i, v in enumerate(top10['Test R²']):
    axes[0].text(v + 0.01, i, f'{v:.3f}', va='center', fontweight='bold')

# Test RMSE
axes[1].barh(range(len(top10)), top10['Test RMSE'], color=colors)
axes[1].set_yticks(range(len(top10)))
axes[1].set_yticklabels(top10['Model'])
axes[1].set_xlabel('Test RMSE ($)')
axes[1].set_title('Test RMSE (lower = better)', fontweight='bold')
axes[1].invert_yaxis()
for i, v in enumerate(top10['Test RMSE']):
    axes[1].text(v + 50, i, f'${v:,.0f}', va='center', fontweight='bold')

# Test MAE
axes[2].barh(range(len(top10)), top10['Test MAE'], color=colors)
axes[2].set_yticks(range(len(top10)))
axes[2].set_yticklabels(top10['Model'])
axes[2].set_xlabel('Test MAE ($)')
axes[2].set_title('Test MAE (lower = better)', fontweight='bold')
axes[2].invert_yaxis()
for i, v in enumerate(top10['Test MAE']):
    axes[2].text(v + 50, i, f'${v:,.0f}', va='center', fontweight='bold')

plt.suptitle('Top 10 Models — Performance Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9.7 Compare Best Automated Model vs Manual Models

In [ ]:
# Load manual model results
manual_results = pd.read_csv('/content/drive/MyDrive/model_results.csv', index_col=0)

# Add best automated model results
auto_row = pd.DataFrame({
    'Test R²': [best_test_r2],
    'Test RMSE': [best_test_rmse],
    'Test MAE': [best_test_mae]
}, index=[f'Best Auto ({best_model_name})'])

# Combine
final_comparison = pd.concat([manual_results[['Test R²', 'Test RMSE', 'Test MAE']], auto_row])

print("\n" + "=" * 60)
print("  MANUAL MODELS vs BEST AUTOMATED MODEL")
print("=" * 60)
final_comparison

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6']
model_labels = final_comparison.index.tolist()

# R² comparison
axes[0].bar(range(len(model_labels)), final_comparison['Test R²'],
            color=colors[:len(model_labels)])
axes[0].set_title('Test R² Comparison', fontweight='bold')
axes[0].set_ylim(0, 1.1)
axes[0].set_xticks(range(len(model_labels)))
axes[0].set_xticklabels(model_labels, rotation=20, ha='right')
for i, v in enumerate(final_comparison['Test R²']):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# RMSE comparison
axes[1].bar(range(len(model_labels)), final_comparison['Test RMSE'],
            color=colors[:len(model_labels)])
axes[1].set_title('Test RMSE Comparison ($)', fontweight='bold')
axes[1].set_xticks(range(len(model_labels)))
axes[1].set_xticklabels(model_labels, rotation=20, ha='right')
for i, v in enumerate(final_comparison['Test RMSE']):
    axes[1].text(i, v + 100, f'${v:,.0f}', ha='center', fontweight='bold')

plt.suptitle('Manual Models vs Best Automated Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9.8 Save Best Model and Results

In [ ]:
import pickle

# Find and save the best model object
all_models = {**scaled_models, **unscaled_models}
best_model_obj = all_models[best_model_name]

with open('/content/drive/MyDrive/best_auto_model.pkl', 'wb') as f:
    pickle.dump(best_model_obj, f)

# Save the full comparison table
comparison_df.to_csv('/content/drive/MyDrive/auto_model_comparison.csv')

# Save the final comparison (manual vs auto)
final_comparison.to_csv('/content/drive/MyDrive/manual_vs_auto_comparison.csv')

print(f"Best model ({best_model_name}) saved to Google Drive.")
print(f"Comparison tables saved.")

---
## Summary

| Aspect | Detail |
|--------|--------|
| **Models Compared** | 17 regression models |
| **Best Model** | See output above |
| **Test R²** | See output above |
| **Test RMSE** | See output above |
| **vs Manual Models** | Compare RMSE — if similar, our manual work was strong |

**Key insight:** If the best automated model achieves a similar RMSE to our manual XGBoost, it confirms that our feature engineering and manual modeling approach was effective.

**Next step →** Notebook 10: SHAP & LIME Analysis